In [1]:
# ==============================================================================
# EXTERNAL FORGE: ENVIRONMENT HYGIENE
# ==============================================================================
print("🏗️ Preparando cimientos del Colab Externo...")

# Forzamos la actualización de tipos de datos y eliminamos conflictos
!pip install --upgrade ml_dtypes -q
!pip install h3 dtreeviz -q

print("✅ Entorno listo. Procediendo a Montar Drive y Auth.")

🏗️ Preparando cimientos del Colab Externo...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 7.5 MB/s eta 0:00:00
✅ Entorno listo. Procediendo a Montar Drive y Auth.


In [2]:
# ==============================================================================
# CELL 0: PIENZA CLOUD BOOTSTRAP (EXTERNAL FORGE EDITION)
# ==============================================================================
# Environment: Public Google Colab (External Forge)
# Purpose: Manual Authentication to bridge into BigQuery from outside.
# ==============================================================================

# --- 1. CLOUD PASSPORT (OBLIGATORIO EN COLAB EXTERNO) ---
from google.colab import auth, drive
print("🔐 Solicitando Pasaporte de Google Cloud...")
auth.authenticate_user() # Esto abrirá una ventana emergente para que elijas tu cuenta
print("✅ Autenticación de Usuario Exitosa.")

print("\n📂 Montando Bóveda de Drive...")
drive.mount('/content/drive') # Necesario para guardar el modelo al final
print("✅ Drive Sincronizado.")

# --- 2. CORE IMPORTS ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
import warnings
warnings.filterwarnings('ignore')

# --- 3. CLOUD CONNECTIVITY ---
# --- SOVEREIGN COORDINATES ---
PROJECT_ID    = '645009831643'
DATASET_CORE  = 'pienza_mini'

try:
    # El cliente ahora usará tus credenciales de usuario recién validadas
    client = bigquery.Client(project=PROJECT_ID)
    print(f"✅ Cliente BigQuery Activo: {PROJECT_ID}")

    # Prueba de conexión
    probe_query = f"SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_CORE}.offers`"
    client.query(probe_query).result()
    print("✅ Sovereign Data Connection: STABLE.")
except Exception as e:
    print(f"🔴 CRITICAL: Cloud connection failed. Details: {e}")

# --- 4. VISUAL CANON ---
PIENZA_PURPLE = '#440154'
PIENZA_TEAL   = '#21918c'
PIENZA_GREY   = '#FAFAFA'

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': PIENZA_GREY,
    'axes.facecolor': PIENZA_GREY,
    'axes.titlecolor': PIENZA_PURPLE,
    'figure.titlesize': 24,
    'figure.titleweight': 'bold'
})

print("\n--- EXTERNAL FORGE SYSTEM READY ---")

🔐 Solicitando Pasaporte de Google Cloud...
✅ Autenticación de Usuario Exitosa.

📂 Montando Bóveda de Drive...
Mounted at /content/drive
✅ Drive Sincronizado.
✅ Cliente BigQuery Activo: 645009831643
✅ Sovereign Data Connection: STABLE.

--- EXTERNAL FORGE SYSTEM READY ---


In [4]:
# ==============================================================================
# CELL 1: DATA INGESTION (THE RAW STREAM)
# ==============================================================================
print("📡 Iniciando extracción de telemetría desde BigQuery...")

query = f"""
SELECT
    offer_id,
    pickup_lat,
    pickup_lon,
    dropoff_lat,
    dropoff_lon,
    upfront_fare,
FROM `{PROJECT_ID}.{DATASET_CORE}.offers`
WHERE
    pickup_lat IS NOT NULL
    AND dropoff_lat IS NOT NULL
"""

try:
    df_raw = client.query(query).to_dataframe()
    print(f"✅ Dataset cargado: {df_raw.shape[0]} viajes.")
    print("   Columnas detectadas:", list(df_raw.columns))
except Exception as e:
    print(f"🔴 Error en Query: {e}")

📡 Iniciando extracción de telemetría desde BigQuery...
✅ Dataset cargado: 4758 viajes.
   Columnas detectadas: ['offer_id', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon', 'upfront_fare']


In [7]:
# ==============================================================================
# CELL 2 (FINAL FIX): THE TOPOLOGICAL FORGE (H3 v4 COMPATIBLE)
# ==============================================================================
import h3
import networkx as nx

print("TB... Forjando topología H3 (API v4)...")

# 1. LIMPIEZA DE TIPOS
cols_coords = ['pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon']
for col in cols_coords:
    df_raw[col] = df_raw[col].astype(float)

# 2. DISCRETIZACIÓN ESPACIAL
H3_RES = 8

def get_hex(lat, lon, res):
    try:
        # SINTAXIS ACTUALIZADA (v4): latlng_to_cell
        return h3.latlng_to_cell(lat, lon, res)
    except Exception as e:
        return None

df_raw['source_hex'] = df_raw.apply(lambda x: get_hex(x['pickup_lat'], x['pickup_lon'], H3_RES), axis=1)
df_raw['target_hex'] = df_raw.apply(lambda x: get_hex(x['dropoff_lat'], x['dropoff_lon'], H3_RES), axis=1)

df_graph_input = df_raw.dropna(subset=['source_hex', 'target_hex'])

# 3. AGREGACIÓN
df_edges = df_graph_input.groupby(['source_hex', 'target_hex']).agg(
    weight=('offer_id', 'count'),
    avg_fare=('upfront_fare', 'mean')
).reset_index()

# 4. CONSTRUCCIÓN DEL GRAFO
G = nx.DiGraph()

for _, row in df_edges.iterrows():
    G.add_edge(
        row['source_hex'],
        row['target_hex'],
        weight=row['weight'],
        fare=row['avg_fare']
    )

print(f"✅ Grafo Construido:")
print(f"   - Nodos (Hexágonos): {G.number_of_nodes()}")
print(f"   - Aristas (Rutas):   {G.number_of_edges()}")

TB... Forjando topología H3 (API v4)...
✅ Grafo Construido:
   - Nodos (Hexágonos): 760
   - Aristas (Rutas):   3530


In [8]:
# ==============================================================================
# CELL 3: ALGORITHMIC INTERROGATION (CENTRALITY)
# ==============================================================================
print("🧠 Ejecutando algoritmos de centralidad...")

# 1. DEGREE (Popularidad / Volumen)
# ¿Qué hexágonos tienen más tráfico total?
degree_dict = dict(G.degree(weight='weight'))

# 2. EIGENVECTOR (Prestigio Económico)
# Usamos 'fare' (Tarifa Promedio) como peso.
# Si conectas con hexágonos de tarifa alta, tu prestigio sube.
try:
    eigen_dict = nx.eigenvector_centrality(G, weight='fare', max_iter=1000)
except:
    # Fallback si el grafo no es conexo (islas aisladas)
    print("⚠️ Eigenvector no convergió, usando Degree como proxy.")
    eigen_dict = degree_dict

# 3. BETWEENNESS (Puentes)
# ¿Qué hexágonos controlan el flujo?
k_sample = min(300, len(G.nodes()))
between_dict = nx.betweenness_centrality(G, weight='weight', k=k_sample)

print("✅ Métricas Calculadas.")

🧠 Ejecutando algoritmos de centralidad...
✅ Métricas Calculadas.


In [10]:
# ==============================================================================
# CELL 4: KEPLER.GL EXPORT PROTOCOL -> DRIVE
# ==============================================================================
import os

# 1. DEFINICIÓN DE RUTA DE SALIDA
EXPORT_PATH = '/content/drive/MyDrive/_Pienza/Assets/Phase_4'

# Creamos la carpeta si no existe (Seguridad)
os.makedirs(EXPORT_PATH, exist_ok=True)
print(f"📂 Ruta de exportación verificada: {EXPORT_PATH}")

print("🚀 Generando artefactos para Kepler.gl...")

# --- A. PREPARAR NODOS (Puntos Hexágono) ---
nodes_data = []
for node in G.nodes():
    try:
        # SINTAXIS H3 v4: cell_to_latlng
        lat, lon = h3.cell_to_latlng(node)

        nodes_data.append({
            'h3_id': node,
            'lat': lat,
            'lon': lon,
            'degree_volumen': degree_dict.get(node, 0),      # Popularidad (Tamaño)
            'eigen_prestigio': eigen_dict.get(node, 0),      # Calidad Económica (Color)
            'between_puente': between_dict.get(node, 0)      # Control de Flujo
        })
    except:
        continue

df_nodes_export = pd.DataFrame(nodes_data)

# --- B. PREPARAR ARCOS (Rutas) ---
edges_data = []
for u, v, data in G.edges(data=True):
    try:
        # SINTAXIS H3 v4
        lat_u, lon_u = h3.cell_to_latlng(u)
        lat_v, lon_v = h3.cell_to_latlng(v)

        edges_data.append({
            'source_lat': lat_u, 'source_lon': lon_u,
            'target_lat': lat_v, 'target_lon': lon_v,
            'viajes_volumen': data['weight'],    # Grosor
            'tarifa_promedio': data['fare']      # Color
        })
    except:
        continue

df_edges_export = pd.DataFrame(edges_data)

# --- C. GUARDAR DIRECTO EN DRIVE ---
node_file = f'{EXPORT_PATH}/opus_nodos.csv'
edge_file = f'{EXPORT_PATH}/opus_arcos.csv'

df_nodes_export.to_csv(node_file, index=False)
df_edges_export.to_csv(edge_file, index=False)

print(f"✅ ¡Misión Cumplida! Archivos guardados en tu Bóveda Pienza:")
print(f"   1. {node_file} ({len(df_nodes_export)} zonas)")
print(f"   2. {edge_file} ({len(df_edges_export)} rutas)")
print("\n👉 Ya puedes ir a Kepler.gl e importar directamente desde tu Drive o descargar los archivos.")

📂 Ruta de exportación verificada: /content/drive/MyDrive/_Pienza/Assets/Phase_4
🚀 Generando artefactos para Kepler.gl...
✅ ¡Misión Cumplida! Archivos guardados en tu Bóveda Pienza:
   1. /content/drive/MyDrive/_Pienza/Assets/Phase_4/opus_nodos.csv (760 zonas)
   2. /content/drive/MyDrive/_Pienza/Assets/Phase_4/opus_arcos.csv (3530 rutas)

👉 Ya puedes ir a Kepler.gl e importar directamente desde tu Drive o descargar los archivos.
